# Portfolio Bot — Split Allocation ($20 BTC + $10 BNB)

**Strategy:** $30/mo total injection. $20 BTC base + $10 BNB base on red candles.
45% sell at ATH (risk-optimal). BTC: 1%+2%->50%, BNB: 1%+1%->40%

In [ ]:
import pandas as pd, requests, time, numpy as np, matplotlib.pyplot as plt, matplotlib.dates as mdates
from datetime import datetime, timezone
pd.set_option('display.max_columns', None); pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
def fetch_klines(s):
    url = 'https://api.binance.com/api/v3/klines'; all_k = []; st = 0
    while True:
        p = {'symbol': s, 'interval': '1M', 'startTime': st, 'limit': 500}
        d = requests.get(url, params=p).json()
        if not d: break
        all_k.extend(d)
        if len(d) < 500: break
        st = d[-1][0] + 1; time.sleep(0.1)
    return all_k
cols = ['ts','o','h','l','c','v','ct','qv','t','tbb','tbq','ig']
btc = pd.DataFrame(fetch_klines('BTCUSDT'), columns=cols); btc['date'] = pd.to_datetime(btc['ts'], unit='ms', utc=True)
bnb = pd.DataFrame(fetch_klines('BNBUSDT'), columns=cols); bnb['date'] = pd.to_datetime(bnb['ts'], unit='ms', utc=True)
for c in ['o','h','l','c','v']: btc[c]=btc[c].astype(float); bnb[c]=bnb[c].astype(float)
btc = btc[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
bnb = bnb[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
s = max(btc['date'].min(), bnb['date'].min()); e = min(btc['date'].max(), bnb['date'].max())
btc = btc[(btc['date']>=s)&(btc['date']<=e)].reset_index(drop=True)
bnb = bnb[(bnb['date']>=s)&(bnb['date']<=e)].reset_index(drop=True)
print(f'BTC: {btc["date"].min().strftime("%Y-%m")} -> {btc["date"].max().strftime("%Y-%m")} ({len(btc)} months)')
print(f'BNB: {bnb["date"].min().strftime("%Y-%m")} -> {bnb["date"].max().strftime("%Y-%m")} ({len(bnb)} months)')

In [ ]:
SELL = 0.45; USDT = 0.0; INJ = 0.0
bh = 0.0; bhi = 0.0; br = 1.0; bi = 0.0
nh = 0.0; nhi = 0.0; nr = 1.0; ni = 0.0
rec = []
for i in range(len(btc)):
    b = btc.iloc[i]; n = bnb.iloc[i]
    bc = b['c']; nc = n['c']; d = b['date']; act = []
    USDT += 30.0; INJ += 30.0
    # BTC buy: $20 base
    if bc < b['o']:
        ext = USDT * (br/100.0) if USDT > 0 else 0.0
        sp = min(20.0 + ext, USDT)
        bh += sp / bc; bi += 20.0; USDT -= sp
        act.append(f'BTC buy ${sp:.0f} @ {bc:.0f}')
    # BNB buy: $10 base
    if nc < n['o']:
        ext = USDT * (nr/100.0) if USDT > 0 else 0.0
        sp = min(10.0 + ext, USDT)
        nh += sp / nc; ni += 10.0; USDT -= sp
        act.append(f'BNB buy ${sp:.0f} @ {nc:.0f}')
    # BTC sell at new ATH
    pp = bhi
    if bc > bhi: bhi = bc
    if bh > 0.000001 and bc > pp:
        s = bh * SELL; USDT += s * bc; bh -= s; br = 1.0
        act.append(f'BTC sell {SELL*100:.0f}% @ {bc:.0f}')
    elif br < 50: br = min(50, br + 2)
    # BNB sell at new ATH
    qq = nhi
    if nc > nhi: nhi = nc
    if nh > 0.000001 and nc > qq:
        s = nh * SELL; USDT += s * nc; nh -= s; nr = 1.0
        act.append(f'BNB sell {SELL*100:.0f}% @ {nc:.0f}')
    elif nr < 40: nr = min(40, nr + 1)
    rec.append({'date': d, 'bc': bc, 'nc': nc,
        'bh': bh, 'bv': bh*bc, 'br': br,
        'nh': nh, 'nv': nh*nc, 'nr': nr,
        'usdt': USDT, 'inj': INJ,
        'pf': bh*bc + nh*nc + USDT,
        'act': ' | '.join(act) if act else 'nothing',
        'bi': bi, 'ni': ni})
res = pd.DataFrame(rec)
print(f'Months: {len(res)} | Buy: {len(res[res["act"].str.contains("buy", na=False)])} | Sell: {len(res[res["act"].str.contains("sell", na=False)])}')

In [ ]:
f = res.iloc[-1]
eq = res['pf']
ddr = (eq.cummax() - eq) / eq.cummax(); maxdd = ddr.max() * 100
m = eq.pct_change().dropna(); m = m[m.index >= 12]
sharpe = (m.mean()/m.std())*np.sqrt(12) if m.std()>0 else 0
p = m[m>0].sum(); ne = m[m<0].abs().sum(); pf = p/ne if ne>0 else float('inf')
ann = (1+(f['pf']/f['inj']-1))**(12/len(res))-1
calmar = ann/(maxdd/100) if maxdd>0 else 0
btc_r = (f['bv']/f['bi']-1)*100 if f['bi']>0 else 0
bnb_r = (f['nv']/f['ni']-1)*100 if f['ni']>0 else 0
print('='*72)
print('PORTFOLIO BOT — $20 BTC + $10 BNB ($30/mo total)')
print('='*72)
print(f'Period:         {res["date"].min().strftime("%Y-%m")} -> {res["date"].max().strftime("%Y-%m")} ({len(res)} mo)')
print(f'Injection:      $30/mo = ${f["inj"]:.0f} total')
print()
print('--- ASSET BREAKDOWN ---')
print(f'BTC base:       ${f["bi"]:.0f} (${f["bi"]/len(res):.2f}/mo)')
print(f'BTC held:       {f["bh"]:.8f} (${f["bv"]:.2f} @ {f["bc"]:.0f})')
print(f'BTC return:     {btc_r:.1f}%')
print(f'BNB base:       ${f["ni"]:.0f} (${f["ni"]/len(res):.2f}/mo)')
print(f'BNB held:       {f["nh"]:.8f} (${f["nv"]:.2f} @ {f["nc"]:.0f})')
print(f'BNB return:     {bnb_r:.1f}%')
print()
print('--- PORTFOLIO ---')
print(f'USDT reserve:   ${f["usdt"]:.2f}')
print(f'Portfolio:      ${f["pf"]:.2f}')
print(f'Total P&L:      ${f["pf"]-f["inj"]:.2f}')
print(f'Return:         {(f["pf"]/f["inj"]-1)*100:.2f}%')
print(f'Max DD:         {maxdd:.2f}%')
print(f'Sharpe:         {sharpe:.2f}')
print(f'PF:             {pf:.2f}')
print(f'Calmar:         {calmar:.2f}')
print()
print('--- CASHFLOW ---')
print(f'Total injected: ${f["inj"]:.0f}')
print(f'BTC base:       ${f["bi"]:.0f}')
print(f'BNB base:       ${f["ni"]:.0f}')
print(f'Reinvested:     ${f["inj"]-f["bi"]-f["ni"]:.0f}')
print(f'Reserve left:   ${f["usdt"]:.2f}')

In [ ]:
# Dashboard chart
fig = plt.figure(figsize=(16, 14))
gs = fig.add_gridspec(5, 2, height_ratios=[3, 1, 1.2, 1.2, 1.2])
ax = fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['inj'], res['pf'],
    where=res['pf']>=res['inj'], color='green', alpha=0.12, label='Profit')
ax.fill_between(res['date'], res['pf'], res['inj'],
    where=res['pf']<res['inj'], color='red', alpha=0.12, label='Loss')
ax.plot(res['date'], res['inj'], color='gray', lw=1.5, ls='--', label='Total Injected ($30/mo)')
ax.plot(res['date'], res['pf'], color='#2563eb', lw=2, label='Portfolio')
buys = res[res['act'].str.contains('buy', na=False)]
sells = res[res['act'].str.contains('sell', na=False)]
ax.scatter(buys['date'], buys['pf'], color='#16a34a', s=20, marker='^', zorder=5, alpha=0.6, label='Buy')
ax.scatter(sells['date'], sells['pf'], color='#dc2626', s=40, marker='v', zorder=5, label='Sell 45%')
ax.set_title('Portfolio Bot — $20 BTC + $10 BNB ($30/mo, 45% sell at ATH)', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=8, ncol=3); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, ddr*100, color='#dc2626', alpha=0.35)
ax.plot(res['date'], ddr*100, color='#dc2626', lw=0.8)
ax.axhline(y=maxdd, color='#991b1b', ls=':', lw=1, label=f'Max DD: {maxdd:.1f}%')
ax.set_ylabel('DD (%)'); ax.set_ylim(bottom=0); ax.legend(fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[2, 0])
ax.fill_between(res['date'], 0, res['bv'], color='#f59e0b', alpha=0.3)
ax.plot(res['date'], res['bv'], color='#d97706', lw=1.5)
ax.set_ylabel('BTC Value'); ax.set_title(f'BTC ($20 base, {btc_r:.0f}%)'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[2, 1])
ax.step(res['date'], res['br'], where='post', color='#f59e0b', lw=1.5)
ax.axhline(y=50, color='#f59e0b', ls=':', alpha=0.5, label='Cap: 50%')
ax.set_ylabel('Reinvest %'); ax.set_ylim(0, 58); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[3, 0])
ax.fill_between(res['date'], 0, res['nv'], color='#8b5cf6', alpha=0.3)
ax.plot(res['date'], res['nv'], color='#7c3aed', lw=1.5)
ax.set_ylabel('BNB Value'); ax.set_title(f'BNB ($10 base, {bnb_r:.0f}%)'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[3, 1])
ax.step(res['date'], res['nr'], where='post', color='#8b5cf6', lw=1.5)
ax.axhline(y=40, color='#8b5cf6', ls=':', alpha=0.5, label='Cap: 40%')
ax.set_ylabel('Reinvest %'); ax.set_ylim(0, 46); ax.legend(fontsize=8); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[4, 0])
ax.fill_between(res['date'], 0, res['usdt'], color='#10b981', alpha=0.3)
ax.plot(res['date'], res['usdt'], color='#059669', lw=1.5)
ax.set_ylabel('USDT Reserve'); ax.set_title('Cash Ready for Bear'); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax = fig.add_subplot(gs[4, 1])
ax.plot(res['date'], res['bc'], color='#f59e0b', lw=1, alpha=0.7, label='BTC')
axb = ax.twinx()
axb.plot(res['date'], res['nc'], color='#8b5cf6', lw=1, alpha=0.7, label='BNB')
ax.set_ylabel('BTC (USDT)'); axb.set_ylabel('BNB (USDT)')
ax.set_title('Prices'); ax.grid(True, alpha=0.25)
ax.legend(loc='upper left', fontsize=8); axb.legend(loc='upper right', fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-split.png', dpi=150, bbox_inches='tight')
print('Saved dca-bt-buy-bot-portfolio-split.png')
plt.show()

In [ ]:
# Stacked allocation
fig, ax = plt.subplots(figsize=(14, 7))
ax.stackplot(res['date'], res['bv'], res['nv'], res['usdt'],
    labels=['BTC', 'BNB', 'USDT Reserve'], colors=['#f59e0b','#8b5cf6','#10b981'], alpha=0.8)
ax.plot(res['date'], res['inj'], color='gray', lw=1.5, ls='--', label='Total Injected')
ax.set_title('Portfolio Composition — $20 BTC + $10 BNB/mo', fontsize=13, fontweight='bold')
ax.set_ylabel('USDT'); ax.legend(loc='upper left', fontsize=9); ax.grid(True, alpha=0.25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y')); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-portfolio-split-stacked.png', dpi=150, bbox_inches='tight')
print('Saved dca-bt-buy-bot-portfolio-split-stacked.png')
plt.show()

In [ ]:
# Monthly cashflow table
cf = []
for i, row in res.iterrows():
    bb = 0; nb = 0; bs = 0; ns = 0
    if 'BTC buy' in row['act']:
        import re
        m = re.search(r'BTC buy \$(\d+)', row['act'])
        if m: bb = float(m.group(1))
    if 'BNB buy' in row['act']:
        m = re.search(r'BNB buy \$(\d+)', row['act'])
        if m: nb = float(m.group(1))
    if i > 0:
        bd = row['bh'] - res.iloc[i-1]['bh']
        if bd < -0.000001: bs = abs(bd)*(row['bc']+res.iloc[i-1]['bc'])/2
        nd = row['nh'] - res.iloc[i-1]['nh']
        if nd < -0.000001: ns = abs(nd)*(row['nc']+res.iloc[i-1]['nc'])/2
    cf.append({'d': row['date'], 'bb': bb, 'nb': nb, 'bs': bs, 'ns': ns,
        'usdt': row['usdt'], 'pf': row['pf']})
cf = pd.DataFrame(cf)
print(f"{'Date':>10s} {'BTCbuy':>8s} {'BNBbuy':>8s} {'BTCsell':>8s} {'BNBsell':>8s} {'Resv':>8s} {'Portfolio':>10s}")
print('-'*66)
for _, r in cf.iterrows():
    print(f"{r['d'].strftime('%Y-%m'):>10s} {r['bb']:>8.0f} {r['nb']:>8.0f} {r['bs']:>8.1f} {r['ns']:>8.1f} {r['usdt']:>8.1f} {r['pf']:>10.1f}")
print()
print(f'Total BTC buys:  ${cf["bb"].sum():.0f}')
print(f'Total BNB buys:  ${cf["nb"].sum():.0f}')
print(f'Total BTC sells: ${cf["bs"].sum():.0f}')
print(f'Total BNB sells: ${cf["ns"].sum():.0f}')
print(f'Final reserve:   ${cf["usdt"].iloc[-1]:.0f}')
print(f'Final portfolio: ${cf["pf"].iloc[-1]:.0f}')